# 06 — Transformer Models

Purpose: Evaluate BatteryGPT, TFT, iTransformer, and physics-informed iTransformer.

Leakage rule: current-cycle full discharged capacity and SOH are labels only.
Preprocessing is fitted on training batteries, validation batteries control
selection and early stopping, and test batteries are scored exactly once.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "v3"
RESULTS = ARTIFACT_ROOT / "results"
FIGURES = ARTIFACT_ROOT / "figures"
FEATURES = ARTIFACT_ROOT / "features"
for path in (RESULTS, FIGURES, FEATURES):
    path.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")


Project root: E:\VIT\aiBatteryLifecycle\application


In [2]:
import numpy as np, pandas as pd
from src.experiments.deep import run_grouped_sequence_benchmark
data = np.load(FEATURES / "nasa" / "sequences.npz")
index = pd.read_csv(FEATURES / "nasa" / "sequence_index.csv")
models = ("battery_gpt", "temporal_fusion_transformer", "itransformer", "physics_itransformer")
metric_path = RESULTS / "nasa_transformer_fold_metrics.csv"
prediction_path = RESULTS / "nasa_transformer_predictions.csv"
if not metric_path.exists() or not prediction_path.exists():
    metrics, predictions = run_grouped_sequence_benchmark(data["X"], index, dataset_name="NASA", model_ids=models)
    metrics.to_csv(metric_path, index=False)
    predictions.to_csv(prediction_path, index=False)
else:
    metrics = pd.read_csv(metric_path)
display(metrics.groupby("model")[["mae", "rmse", "r2", "within_5pp"]].mean().sort_values("mae"))

,mae,rmse,r2,within_5pp
model,,,,
itransformer,7.295363,9.846966,-1.252178,0.569976
physics_itransformer,7.532534,10.068538,-0.996817,0.537726
temporal_fusion_transformer,7.657566,10.480169,-0.953901,0.525387
battery_gpt,8.475195,11.293164,-1.901130,0.513108
